# DOC-02 insurance application: extraction test (Qwen2.5-VL)

Run on a Colab T4 GPU (Runtime > Change runtime type). Restart the session first, then run every cell in order from the top. The check cell must print `OK`.

Output: `doc02_results.json`.

In [ ]:
!rm -rf /content/repo
!git clone -q -b feature/doc02-insurance-rules https://github.com/Devadn11206/DOC_Extractor_QUAPT.git /content/repo
!cd /content/repo && git log --oneline -1

In [ ]:
import subprocess
print('Code version:', subprocess.run(['git','-C','/content/repo','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())
src = open('/content/repo/src/doc_extractor/insurance_reader.py', encoding='utf-8').read()
assert 'def read_insurance_form' in src, 'OLD CODE: restart the session and re-run the clone cell.'
assert 'EVIDENCE_NOT_CHECKED' in open('/content/repo/src/doc_extractor/insurance_form.py', encoding='utf-8').read(), 'OLD CODE: restart the session and re-run the clone cell.'
print('OK: DOC-02 code present')

In [ ]:
!pip install -q -r /content/repo/requirements.txt rapidocr_onnxruntime accelerate
import transformers, torch
print('transformers', transformers.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
# If transformers is below 5.0: !pip install -q "transformers>=5", then Runtime > Restart session and run from the top.

In [ ]:
import sys, types
pkg = types.ModuleType('doc_extractor')
pkg.__path__ = ['/content/repo/src/doc_extractor']
sys.modules['doc_extractor'] = pkg
from doc_extractor.casa_reader import QwenVLCasaReader
from doc_extractor.insurance_reader import read_insurance_form
print('DOC-02 modules loaded')

In [ ]:
from google.colab import files
import os
os.makedirs('/content/doc02', exist_ok=True)
for name, data in files.upload().items():
    open(os.path.join('/content/doc02', name), 'wb').write(data)
print('uploaded to /content/doc02')

In [ ]:
import os, json, glob
from PIL import Image

images = sorted(glob.glob('/content/doc02/*.jpg') + glob.glob('/content/doc02/*.jpeg') + glob.glob('/content/doc02/*.png'))
print(len(images), 'images')
reader = QwenVLCasaReader()
results = {}
for p in images:
    r = read_insurance_form(Image.open(p), reader)
    results[os.path.basename(p)] = r
    print('==', os.path.basename(p), '| status:', r['status'], '| model secs:', r['timing']['model_seconds'])
    for k, v in r['fields'].items():
        print(f"   {k:26s} {str(v['value']):28s} {v['null_reason'] or '':9s} {'REVIEW' if v['review_required'] else 'ok':6s} {v['error'] or ''}")
    for n in r['nominees']:
        print('   nominee', n)
    print('   flags', [f['code'] for f in r['flags']])
json.dump(results, open('/content/doc02/doc02_results.json', 'w'), indent=2, ensure_ascii=False, default=str)

In [ ]:
from google.colab import files
files.download('/content/doc02/doc02_results.json')